# Denoising Diffusion Probabilistic Models (DDPM)
"Denoising Diffusion Probabilistic Models" (Ho, Jain, Abbeel, 2020)

**Idea:** slowly destroy an image by adding Gaussian noise over $T$ steps, then train a network to **predict the noise** that was added. To generate, start from pure noise and remove it one small step at a time.

### DDPM vs previous notebooks
| | Classifier (CNN / ViT) | Autoencoder | DDPM |
|---|---|---|---|
| Task | discriminative: image -> label | reconstruction: image -> same image | **generative**: noise -> new image |
| Labels needed | yes | no | no |
| Model input | clean image | clean image | **noisy image $x_t$ + timestep $t$** |
| Model predicts | class logits `(N, 10)` | the image `(N, C, H, W)` | **the noise $\epsilon$** that was added `(N, C, H, W)` |
| Architecture | CNN / Transformer encoder -> head | encoder -> bottleneck -> decoder | **UNet** (encoder-decoder with skip connections) + time embedding |
| Loss | cross-entropy | MSE(output, image) | **MSE(predicted noise, true noise)** |
| Inference | 1 forward pass | 1 forward pass | **$T = 1000$ forward passes** (iterative denoising) |
| Position/time info | ViT: learned pos-embedding | - | sinusoidal **timestep** embedding (same formula as Transformer positions) |

Shape flow (one training step): `x_0 (N, 1, 32, 32) + noise -> x_t (N, 1, 32, 32), t (N,) -> UNet -> eps_pred (N, 1, 32, 32) -> MSE with eps`

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## 1. Noise schedule
$T = 1000$ steps, $\beta_t$ grows **linearly** from $10^{-4}$ to $0.02$ (the paper's values). Small steps early keep fine detail; by $t = T$ the image is pure noise.

Everything is precomputed once as 1-D tensors indexed by $t$ (0-indexed in code: `t = 0..T-1`).

### Formula: Noise schedule
$$\beta_t = \beta_1 + \frac{t-1}{T-1}(\beta_T - \beta_1), \qquad \alpha_t = 1 - \beta_t, \qquad \bar\alpha_t = \prod_{s=1}^{t} \alpha_s$$
- $t \in \{1, \dots, T\}$: timestep (code uses `t = 0..T-1`, i.e. code index = $t - 1$)
- $T$: number of diffusion steps (1000)
- $\beta_t$: variance of the noise added at step $t$; linear from $\beta_1 = 10^{-4}$ to $\beta_T = 0.02$
- $\alpha_t$: fraction of signal variance kept at step $t$
- $\bar\alpha_t$: fraction of signal variance kept after $t$ steps ($\bar\alpha_1 \approx 0.9999$, $\bar\alpha_T \approx 4 \times 10^{-5}$)

In [ ]:
T = 1000
betas = torch.linspace(1e-4, 0.02, T, device=device)   # (T,)
alphas = 1.0 - betas                                     # (T,)
alpha_bars = torch.cumprod(alphas, dim=0)                # (T,)

print(f"alpha_bar: t=0 {alpha_bars[0]:.4f} | t=T/2 {alpha_bars[T//2]:.4f} | t=T-1 {alpha_bars[-1]:.6f}")

plt.figure(figsize=(5, 3))
plt.plot(alpha_bars.cpu(), label=r"$\bar\alpha_t$ (signal)")
plt.plot((1 - alpha_bars).cpu(), label=r"$1-\bar\alpha_t$ (noise)")
plt.xlabel("t"); plt.legend(); plt.show()

## 2. Forward process $q(x_t \mid x_0)$
Adding noise step by step is a chain of Gaussians, but it collapses to **one closed-form jump** (see the formulas below).

So during training we can jump to any random $t$ directly - no loop over steps. There are **no learned parameters** here.

### Formula: Forward process (one step)
$$q(x_t \mid x_{t-1}) = \mathcal{N}\!\left(x_t;\ \sqrt{1-\beta_t}\,x_{t-1},\ \beta_t I\right)$$
- $x_0$: clean image ($1 \times 32 \times 32$, pixels in $[-1, 1]$); $x_t$: image after $t$ noising steps
- $\mathcal{N}(x; \mu, \Sigma)$: Gaussian density of $x$ with mean $\mu$, covariance $\Sigma$
- $I$: identity matrix (noise is independent per pixel)
- $\sqrt{1-\beta_t}$ scales the signal down so the total variance stays ~1 (it doesn't blow up)
- fixed, no learned parameters

### Formula: Forward process (closed form)
$$q(x_t \mid x_0) = \mathcal{N}\!\left(x_t;\ \sqrt{\bar\alpha_t}\,x_0,\ (1-\bar\alpha_t) I\right) \quad\Longleftrightarrow\quad x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$
- follows from composing Gaussians: variances of independent noise add up
- $\epsilon$: standard Gaussian noise, same shape as $x_0$ - this is the model's regression target
- $\sqrt{\bar\alpha_t}$: signal coefficient; $\sqrt{1-\bar\alpha_t}$: noise coefficient (`q_sample`)

In [ ]:
def q_sample(x0, t, noise):
    # x0: (N, C, H, W), t: (N,), noise: (N, C, H, W) -> x_t: (N, C, H, W)
    a_bar = alpha_bars[t].view(-1, 1, 1, 1)  # (N, 1, 1, 1) to broadcast over pixels
    return a_bar.sqrt() * x0 + (1 - a_bar).sqrt() * noise

## 3. Timestep embedding
The same UNet is used for every $t$, so it must be told **how noisy** the input is. $t$ is turned into a vector with the **sinusoidal encoding from the Transformer** ("Attention Is All You Need") - there it encoded a token's *position*, here it encodes the *timestep* (formula below).

(Only cosmetic difference: we concatenate `[sin, cos]` instead of interleaving them.) An MLP on top turns it into the embedding that each residual block receives.

### Formula: Sinusoidal timestep embedding
$$PE(t)_{i} = \sin\!\left(\frac{t}{10000^{2i/d}}\right), \qquad PE(t)_{d/2 + i} = \cos\!\left(\frac{t}{10000^{2i/d}}\right), \qquad i = 0, \dots, d/2 - 1$$
- $t$: timestep (integer, code index 0..999)
- $d$: embedding size (`ch` = 64)
- $10000^{2i/d}$: wavelengths from $2\pi$ to $\approx 10000 \cdot 2\pi$ - low $i$ changes fast with $t$, high $i$ slowly, so every $t$ gets a unique, smooth code
- identical to the Transformer positional encoding (which interleaves $\sin$/$\cos$ at even/odd indices; we concatenate them - same information)
- followed by $\text{Linear}(64 \to 256) \to \text{SiLU} \to \text{Linear}(256 \to 256)$, then a per-block $\text{Linear}(256 \to C_\text{out})$ added to the feature map

In [ ]:
class SinusoidalEmbedding(nn.Module):
    # (N,) -> (N, dim)
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)  # (half,) = 1 / 10000^(2i/d)
        args = t.float()[:, None] * freqs[None, :]                                          # (N, half)
        return torch.cat([args.sin(), args.cos()], dim=1)                                   # (N, dim)

## 4. UNet building blocks
- **ResBlock**: `GroupNorm -> SiLU -> Conv` twice, with a residual connection (like ResNet). The time embedding is projected to `out_ch` and **added to every pixel** of the feature map between the two convs - that's how $t$ is injected.
- **GroupNorm** instead of BatchNorm: the batch mixes very different noise levels, so batch statistics are meaningless.
- **SelfAttention**: plain single-head attention over the $H \times W$ pixels (each pixel = a token). Only used at the lowest resolution (8x8 = 64 tokens) where it's cheap.
- **Down / Up**: stride-2 conv to halve, nearest-neighbor upsample + conv to double.

### Formula: GroupNorm
$$\text{GN}(x) = \gamma \odot \frac{x - \mu_g}{\sqrt{\sigma_g^2 + \epsilon}} + \beta$$
- $x$: feature map of one sample, channels split into $G = 8$ groups
- $\mu_g, \sigma_g^2$: mean and variance over the channels of group $g$ and all $H \times W$ pixels (per sample - independent of the batch)
- $\epsilon$: stability constant ($10^{-5}$); $\gamma, \beta$: learned per-channel scale and shift (this $\beta$ is unrelated to the noise schedule $\beta_t$)

### Formula: SiLU (Swish)
$$\text{SiLU}(x) = x\,\sigma(x) = \frac{x}{1 + e^{-x}}$$
- $x$: scalar input (applied element-wise); $\sigma$: logistic sigmoid
- smooth, non-monotonic ReLU-like activation (close to GELU)

### Formula: Self-attention (at 8x8)
$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{C}}\right)V$$
- tokens: the $HW = 64$ pixels of the $8 \times 8$ feature map, each a $C = 128$-dim vector
- $Q, K, V \in \mathbb{R}^{64 \times 128}$: from a $1\times1$ conv (a per-pixel linear layer); single head
- $\sqrt{C}$: keeps dot-product variance ~1 so softmax doesn't saturate
- output goes through another $1\times1$ conv and is added back residually

In [ ]:
class ResBlock(nn.Module):
    # x: (N, in_ch, H, W), t_emb: (N, time_dim) -> (N, out_ch, H, W)
    def __init__(self, in_ch, out_ch, time_dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.time_proj = nn.Linear(time_dim, out_ch)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()  # match channels for the residual

    def forward(self, x, t_emb):
        h = self.conv1(F.silu(self.norm1(x)))                    # (N, out_ch, H, W)
        h = h + self.time_proj(F.silu(t_emb))[:, :, None, None]  # add (N, out_ch, 1, 1) -> same shift on every pixel
        h = self.conv2(self.dropout(F.silu(self.norm2(h))))    # (N, out_ch, H, W)
        return h + self.skip(x)                                  # skip: (N, in_ch, H, W) -> (N, out_ch, H, W)


class SelfAttention(nn.Module):
    # (N, C, H, W) -> (N, C, H, W)
    def __init__(self, ch):
        super().__init__()
        self.norm = nn.GroupNorm(8, ch)
        self.qkv = nn.Conv2d(ch, 3 * ch, 1)  # 1x1 conv = Linear applied per pixel
        self.proj = nn.Conv2d(ch, ch, 1)

    def forward(self, x):
        N, C, H, W = x.shape
        q, k, v = self.qkv(self.norm(x)).reshape(N, 3, C, H * W).unbind(1)  # each (N, C, HW)
        att = (q.transpose(1, 2) @ k) / C ** 0.5                             # (N, HW, HW)
        att = att.softmax(dim=-1)
        out = (v @ att.transpose(1, 2)).reshape(N, C, H, W)                  # (N, C, HW) -> (N, C, H, W)
        return x + self.proj(out)                                            # (N, C, H, W)


class Downsample(nn.Module):
    # (N, C, H, W) -> (N, C, H/2, W/2)
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, stride=2, padding=1)

    def forward(self, x):
        return self.conv(x)


class Upsample(nn.Module):
    # (N, C, H, W) -> (N, C, 2H, 2W)
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, padding=1)

    def forward(self, x):
        return self.conv(F.interpolate(x, scale_factor=2, mode="nearest"))

## 5. UNet
Encoder `32 -> 16 -> 8`, a middle block with attention at 8x8, then a decoder `8 -> 16 -> 32`. Like an autoencoder, but each decoder level **concatenates the encoder features of the same resolution** (skip connections), so fine detail doesn't have to squeeze through the bottleneck.

Output has the **same shape as the input** - it's the predicted noise $\epsilon_\theta(x_t, t)$.

| level | resolution | channels |
|---|---|---|
| 1 | 32x32 | 64 |
| 2 | 16x16 | 128 |
| 3 / middle | 8x8 | 128 (+ attention) |

In [ ]:
class UNet(nn.Module):
    # x_t: (N, 1, 32, 32), t: (N,) -> eps_pred: (N, 1, 32, 32)
    def __init__(self, in_ch=1, ch=64, time_dim=256):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalEmbedding(ch),
            nn.Linear(ch, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )
        self.conv_in = nn.Conv2d(in_ch, ch, 3, padding=1)

        # encoder
        self.down1 = ResBlock(ch, ch, time_dim)              # 32x32
        self.pool1 = Downsample(ch)
        self.down2 = ResBlock(ch, 2 * ch, time_dim)          # 16x16
        self.pool2 = Downsample(2 * ch)
        self.down3 = ResBlock(2 * ch, 2 * ch, time_dim)      # 8x8

        # middle
        self.mid1 = ResBlock(2 * ch, 2 * ch, time_dim)
        self.mid_attn = SelfAttention(2 * ch)
        self.mid2 = ResBlock(2 * ch, 2 * ch, time_dim)

        # decoder (input channels = previous + skip)
        self.up3 = ResBlock(4 * ch, 2 * ch, time_dim)        # 8x8
        self.unpool2 = Upsample(2 * ch)
        self.up2 = ResBlock(4 * ch, 2 * ch, time_dim)        # 16x16
        self.unpool1 = Upsample(2 * ch)
        self.up1 = ResBlock(3 * ch, ch, time_dim)            # 32x32

        self.norm_out = nn.GroupNorm(8, ch)
        self.conv_out = nn.Conv2d(ch, in_ch, 3, padding=1)

    def forward(self, x, t):
        t_emb = self.time_mlp(t)                         # (N, time_dim)

        x = self.conv_in(x)                              # (N, 64, 32, 32)
        h1 = self.down1(x, t_emb)                        # (N, 64, 32, 32)
        h2 = self.down2(self.pool1(h1), t_emb)           # (N, 128, 16, 16)
        h3 = self.down3(self.pool2(h2), t_emb)           # (N, 128, 8, 8)

        x = self.mid1(h3, t_emb)                         # (N, 128, 8, 8)
        x = self.mid_attn(x)                             # (N, 128, 8, 8)
        x = self.mid2(x, t_emb)                          # (N, 128, 8, 8)

        x = self.up3(torch.cat([x, h3], dim=1), t_emb)   # (N, 256, 8, 8) -> (N, 128, 8, 8)
        x = self.unpool2(x)                              # (N, 128, 16, 16)
        x = self.up2(torch.cat([x, h2], dim=1), t_emb)   # (N, 256, 16, 16) -> (N, 128, 16, 16)
        x = self.unpool1(x)                              # (N, 128, 32, 32)
        x = self.up1(torch.cat([x, h1], dim=1), t_emb)   # (N, 192, 32, 32) -> (N, 64, 32, 32)

        return self.conv_out(F.silu(self.norm_out(x)))   # (N, 1, 32, 32)

model = UNet().to(device)
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
print(model(torch.randn(2, 1, 32, 32, device=device), torch.randint(0, T, (2,), device=device)).shape)

## 6. Data - MNIST
28x28 grayscale digits, 60k train images. Labels are **ignored** (unconditional generation).

- `Pad(2)` -> 32x32 so the UNet can halve the resolution twice cleanly.
- Scale pixels to $[-1, 1]$ so data and the noise prior $\mathcal{N}(0, I)$ live on a similar scale.

Below: one image pushed through the forward process at a few timesteps.

In [ ]:
tf = transforms.Compose([
    transforms.Pad(2),
    transforms.ToTensor(),                 # [0, 1]
    transforms.Normalize((0.5,), (0.5,)),  # [-1, 1]
])
train_ds = datasets.MNIST("../data", train=True, download=True, transform=tf)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)

x0 = train_ds[0][0][None].to(device)  # (1, 1, 32, 32)
ts = [0, 50, 100, 200, 400, 600, 999]
fig, axes = plt.subplots(1, len(ts), figsize=(2 * len(ts), 2))
for ax, t in zip(axes, ts):
    xt = q_sample(x0, torch.tensor([t], device=device), torch.randn_like(x0))
    ax.imshow(xt[0, 0].cpu().clamp(-1, 1), cmap="gray")
    ax.set_title(f"t={t}"); ax.axis("off")
plt.show()

## 7. Training
Algorithm 1 from the paper - each step:
1. take clean images $x_0$, pick a random $t \sim \text{Uniform}\{1..T\}$ per image, sample noise $\epsilon$
2. make $x_t$ with the closed form (Section 2)
3. loss = MSE between $\epsilon$ and $\epsilon_\theta(x_t, t)$

That's the paper's **simplified loss** $L_\text{simple}$ - a plain regression, no adversarial game (GAN) and no KL term (VAE). Optimizer: AdamW with a constant LR ($2 \times 10^{-4}$, the paper's value).

### Formula: Training loss (simplified objective)
$$L_\text{simple}(\theta) = \mathbb{E}_{t,\,x_0,\,\epsilon}\left[\left\lVert \epsilon - \epsilon_\theta\!\left(\sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon,\ t\right)\right\rVert^2\right]$$
- $\theta$: UNet parameters (~5.9M here)
- $\epsilon_\theta(x_t, t)$: the UNet's predicted noise
- $t \sim \text{Uniform}\{1, \dots, T\}$, $x_0 \sim$ data (MNIST), $\epsilon \sim \mathcal{N}(0, I)$
- $\lVert\cdot\rVert^2$: squared L2 norm; `F.mse_loss` averages it over pixels and batch (batch size 128)
- $\mathbb{E}$ is estimated by the minibatch average

### Formula: AdamW
$$g_t = \nabla_\theta \mathcal{L}(\theta_{t-1})$$
$$m_t = \beta_1 m_{t-1} + (1-\beta_1)\,g_t, \qquad v_t = \beta_2 v_{t-1} + (1-\beta_2)\,g_t^2$$
$$\hat m_t = \frac{m_t}{1-\beta_1^t}, \qquad \hat v_t = \frac{v_t}{1-\beta_2^t}$$
$$\theta_t = \theta_{t-1} - \eta\left(\frac{\hat m_t}{\sqrt{\hat v_t} + \epsilon} + \lambda\,\theta_{t-1}\right)$$
- $t$ here is the optimizer step (not the diffusion timestep); $g_t$: gradient of $L_\text{simple}$
- $m_t, v_t$: EMAs of the gradient and squared gradient; $\hat m_t, \hat v_t$: bias-corrected versions
- $\beta_1, \beta_2$: 0.9, 0.999 (defaults - again unrelated to the noise $\beta_t$)
- $\eta$: learning rate ($2 \times 10^{-4}$, constant); $\epsilon$: $10^{-8}$
- $\lambda$: weight decay (0 here, so this is plain Adam as in the paper)

In [ ]:
epochs = 20
lr = 2e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
losses = []

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for x0, _ in train_loader:                                # labels unused
        x0 = x0.to(device)                                     # (N, 1, 32, 32)
        t = torch.randint(0, T, (x0.shape[0],), device=device) # (N,)
        noise = torch.randn_like(x0)                           # (N, 1, 32, 32)
        xt = q_sample(x0, t, noise)                            # (N, 1, 32, 32)

        loss = F.mse_loss(model(xt, t), noise)  # eps_pred (N, 1, 32, 32) vs noise (N, 1, 32, 32) -> scalar

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        losses.append(loss.item())

    print(f"epoch {epoch+1:2d} | loss {total_loss / len(train_loader):.4f}")

plt.figure(figsize=(5, 3))
plt.plot(losses, alpha=0.4)
plt.xlabel("step"); plt.ylabel("MSE"); plt.yscale("log"); plt.show()

## 8. Sampling (reverse process)
Algorithm 2 from the paper. Start from $x_T \sim \mathcal{N}(0, I)$ and for $t = T, \dots, 1$:

$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\epsilon_\theta(x_t, t)\right) + \sigma_t z, \quad z \sim \mathcal{N}(0, I)$

- the bracket removes the model's estimate of this step's share of the noise
- $\sigma_t z$ adds a little **fresh** noise back (keeps samples diverse); $\sigma_t^2 = \beta_t$ (the paper found $\beta_t$ and $\tilde\beta_t$ work equally well)
- no noise on the very last step

1000 UNet calls per batch - this is why DDPM sampling is slow (later work: DDIM, fewer steps).

### Formula: Reverse process (learned)
$$p_\theta(x_{t-1} \mid x_t) = \mathcal{N}\!\left(x_{t-1};\ \mu_\theta(x_t, t),\ \sigma_t^2 I\right), \qquad p(x_T) = \mathcal{N}(0, I)$$
- $p_\theta$: the generative model, a chain of Gaussians run from $t = T$ down to $t = 1$
- $\mu_\theta$: predicted mean (computed from $\epsilon_\theta$, next formula)
- $\sigma_t^2$: fixed (not learned) variance

### Formula: Posterior $q(x_{t-1} \mid x_t, x_0)$ and the mean parameterization
$$q(x_{t-1} \mid x_t, x_0) = \mathcal{N}\!\left(x_{t-1};\ \tilde\mu_t(x_t, x_0),\ \tilde\beta_t I\right)$$
$$\tilde\mu_t(x_t, x_0) = \frac{\sqrt{\bar\alpha_{t-1}}\,\beta_t}{1-\bar\alpha_t}\,x_0 + \frac{\sqrt{\alpha_t}\,(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}\,x_t, \qquad \tilde\beta_t = \frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}\,\beta_t$$
Substituting $x_0 = \frac{1}{\sqrt{\bar\alpha_t}}\left(x_t - \sqrt{1-\bar\alpha_t}\,\epsilon\right)$ and replacing the true $\epsilon$ with $\epsilon_\theta$:
$$\mu_\theta(x_t, t) = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\epsilon_\theta(x_t, t)\right)$$
- $q(x_{t-1} \mid x_t, x_0)$: the "true" one-step-back distribution, tractable only when $x_0$ is known
- $\tilde\mu_t$, $\tilde\beta_t$: its mean and variance ($\bar\alpha_0 := 1$)
- $\mu_\theta$: what the model uses at sampling time (`mean` in `sample`); training $\epsilon_\theta$ to match $\epsilon$ makes $\mu_\theta$ match $\tilde\mu_t$
- $\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}$: fraction of the predicted noise removed in this step

### Formula: Sampling step
$$x_{t-1} = \mu_\theta(x_t, t) + \sigma_t z, \qquad z \sim \mathcal{N}(0, I) \text{ if } t > 1, \text{ else } z = 0, \qquad \sigma_t^2 = \beta_t$$
- $z$: fresh Gaussian noise each step
- $\sigma_t^2 = \beta_t$ (used here) is the upper bound, optimal when $x_0 \sim \mathcal{N}(0, I)$; $\sigma_t^2 = \tilde\beta_t$ is the lower bound, optimal when $x_0$ is a single point - the paper reports similar results for both
- run for $T = 1000$ steps, 64 samples at once; final output clamped to $[-1, 1]$

In [ ]:
@torch.no_grad()
def sample(model, n, snapshot_every=100):
    # -> x_0: (n, 1, 32, 32), snapshots: list of (t, (n, 1, 32, 32))
    model.eval()
    x = torch.randn(n, 1, 32, 32, device=device)             # x_T
    snapshots = []
    for t in reversed(range(T)):
        t_batch = torch.full((n,), t, device=device, dtype=torch.long)  # (n,)
        eps = model(x, t_batch)                              # (n, 1, 32, 32)

        mean = (x - betas[t] / (1 - alpha_bars[t]).sqrt() * eps) / alphas[t].sqrt()  # (n, 1, 32, 32)
        if t > 0:
            x = mean + betas[t].sqrt() * torch.randn_like(x)
        else:
            x = mean                                         # last step: no noise
        if t % snapshot_every == 0:
            snapshots.append((t, x.clamp(-1, 1).cpu()))
    return x.clamp(-1, 1), snapshots

samples, snapshots = sample(model, 64)

# 8x8 grid of generated digits
fig, axes = plt.subplots(8, 8, figsize=(8, 8))
for ax, img in zip(axes.flatten(), samples.cpu()):
    ax.imshow(img[0], cmap="gray"); ax.axis("off")
plt.suptitle("DDPM samples"); plt.show()

# denoising trajectory of the first 4 samples (left = noisy, right = final)
fig, axes = plt.subplots(4, len(snapshots), figsize=(1.5 * len(snapshots), 6))
for j, (t, snap) in enumerate(snapshots):
    for i in range(4):
        axes[i, j].imshow(snap[i, 0], cmap="gray"); axes[i, j].axis("off")
    axes[0, j].set_title(f"t={t}", fontsize=8)
plt.show()

## Notes
- The model never outputs an image directly - it only ever predicts noise. Generation is 1000 small "subtract a bit of predicted noise, add a bit of fresh noise" steps (this is a form of Langevin dynamics; the paper links $\epsilon$-prediction to **denoising score matching**: $\epsilon_\theta \propto -\nabla_x \log p(x_t)$).
- Why predict $\epsilon$ instead of $x_0$ or $\mu$? The paper tried all three; $\epsilon$-prediction with $L_\text{simple}$ gave the best samples. The target is always unit-variance noise, which makes the regression well-scaled at every $t$.
- $L_\text{simple}$ drops the per-timestep weights of the true variational bound, which **down-weights easy small-$t$ steps** and focuses the network on harder, noisier ones - better samples, slightly worse log-likelihood.
- The paper also keeps an **EMA** of the weights (decay 0.9999) for sampling and uses a bigger UNet (35.7M params, attention at 16x16). Both are skipped here to keep things small; EMA usually gives cleaner samples.
- The loss plateaus quickly (~0.03) even while samples keep improving - the MSE is averaged over all $t$ and is dominated by large-$t$ steps. Judge by the samples, not the loss.
- Conditioning (e.g. on the digit label) is easy: add a label embedding to `t_emb`. Classifier-free guidance builds on exactly that.